# Lokalisierung

06 liefert je Anfrage eine Rangliste von Datenbankbildern; hier wird daraus
eine Koordinate: die Position des besten Treffers. Bezogen auf **alle**
Anfragen, auch die ohne Referenz im Umkreis — eine Koordinate wird immer
geschätzt.

Fünf Aggregationsverfahren (Schwerpunkt, Clustering, Snap, Gated) sind
gemessen und unterliegen Top-1 bei jedem Encoder; sie stehen in
`experiments/localization_aggregation.py`. Grund: Fehlgriffe sind zu 44 %
grobe Verwechslungen, bei denen die Nachbarn geschlossen am falschen Ort
liegen — Konsens bestätigt dann den Fehler.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 150

PROJECT_ROOT = next(d for d in (Path.cwd(), *Path.cwd().parents)
                    if (d / "config.yaml").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import embedding_name, load_config
from src.retrieval import load_retrieval, top_distances
from src.run_guard import (
    code_version,
    embedding_fingerprint,
    print_run_header,
    short_hash,
    validate_config,
)

CFG = load_config(PROJECT_ROOT)
validate_config(CFG)

METHOD = CFG["vpr"]["method"]
ADAPTER = CFG["vpr"].get("adapter", "none")
EMBEDDING_NAME = embedding_name(CFG)
TOP_K = int(CFG["localization"]["top_k"])
UNCERTAIN_RADIUS_M = float(CFG["vpr"]["uncertain_radius_m"])

RESULT_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = RESULT_DIR / "figures" / "localization"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# load_retrieval prueft den Fingerabdruck der Trefferliste.
query_metadata, database_metadata, retrieved_indices, similarities = load_retrieval(
    PROJECT_ROOT, CFG, METHOD, ADAPTER
)
embedding_metadata = pd.read_parquet(
    PROJECT_ROOT / "data" / "embeddings" / METHOD / f"{EMBEDDING_NAME}_metadata.parquet"
)
FINGERPRINT = embedding_fingerprint(CFG, METHOD, ADAPTER, embedding_metadata)

print_run_header(CFG, "08_localization")
print(f"Anfragen:  {len(query_metadata):,}")

## Fehler des besten Treffers

Abstand zwischen echter Position und der Position von Top-1, in Metern.
Dazu zwei triviale Vergleichswerte: ein zufälliges Datenbankbild und der
Stadtmittelpunkt.

In [ ]:
fehler = {}
fehler["Top-1"] = top_distances(query_metadata, database_metadata, retrieved_indices[:, :1])[:, 0]

rng = np.random.default_rng(int(CFG["vpr"]["split_seed"]))
zufall = rng.integers(0, len(database_metadata), len(query_metadata))
fehler["Zufaelliges DB-Bild"] = top_distances(query_metadata, database_metadata, zufall[:, None])[:, 0]

mitte = pd.DataFrame({"lat": [database_metadata["lat"].mean()], "lon": [database_metadata["lon"].mean()]})
fehler["Stadtmittelpunkt"] = top_distances(query_metadata, mitte, np.zeros((len(query_metadata), 1), dtype=int))[:, 0]


def kennzahlen(e):
    return {
        "median_m": float(np.median(e)),
        "p25_m": float(np.percentile(e, 25)),
        "p75_m": float(np.percentile(e, 75)),
        "p90_m": float(np.percentile(e, 90)),
        **{f"unter_{s}m": float((e <= s).mean()) for s in (10, 25, 50, 100)},
    }


tabelle = {name: kennzahlen(e) for name, e in fehler.items()}

kopf = f"{'Verfahren':<26}{'Median':>9}{'p25':>8}{'p75':>9}{'p90':>10}" + \
       "".join(f"{'<' + str(s) + 'm':>8}" for s in (10, 25, 50, 100))
print(kopf)
print("-" * len(kopf))
for name, k in tabelle.items():
    print(f"{name:<26}{k['median_m']:>8.0f}m{k['p25_m']:>7.0f}m{k['p75_m']:>8.0f}m"
          f"{k['p90_m']:>9.0f}m" +
          "".join(f"{k[f'unter_{s}m']:>8.3f}" for s in (10, 25, 50, 100)))

## Struktur der Fehlgriffe

Wie weit liegt ein falscher Top-1 daneben? Beinahetreffer (dieselbe Straße,
ein Stück versetzt) und grobe Verwechslungen (anderer Stadtteil) sind
verschiedene Fehler mit verschiedenen Gegenmitteln. Über die lösbaren
Anfragen zeigt `experiments/confusion_atlas.py` dasselbe auf der Karte.

In [ ]:
falsch = fehler["Top-1"] > UNCERTAIN_RADIUS_M
d_falsch = fehler["Top-1"][falsch]
fehlstruktur = {
    "n_falsch": int(falsch.sum()),
    "anteil_falsch": float(falsch.mean()),
    "median_m": float(np.median(d_falsch)) if falsch.any() else None,
    **{f"unter_{s}m": float((d_falsch <= s).mean()) if falsch.any() else None
       for s in (50, 100, 250, 500, 1000, 2000)},
}
print(f"Falscher Top-1-Treffer (> {UNCERTAIN_RADIUS_M:g} m): "
      f"{fehlstruktur['n_falsch']:,} von {len(fehler['Top-1']):,} Anfragen "
      f"({fehlstruktur['anteil_falsch']:.1%})")
print(f"  Median des Fehlers:  {fehlstruktur['median_m']:,.0f} m")
for s in (50, 100, 500, 1000):
    print(f"  unter {s:>5} m:        {fehlstruktur[f'unter_{s}m']:.1%}")
print("  Die Verteilung ist zweigipflig: knapp daneben oder ein anderes Viertel.")

## Verteilung der Fehler

Die Kurve zeigt, welcher Anteil der Anfragen unter einem gegebenen Fehler
liegt; die grauen Linien sind die Schwellen der Recall-Auswertung.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
schritte = np.logspace(0, 4.5, 200)

for name, e in fehler.items():
    anteil = [(e <= s).mean() for s in schritte]
    stil = "--" if name in ("Zufaelliges DB-Bild", "Stadtmittelpunkt") else "-"
    ax.plot(schritte, anteil, stil, label=name, linewidth=1.6)

for s in (10, 25, 50, 100):
    ax.axvline(s, color="0.85", linewidth=0.8, zorder=0)

ax.set_xscale("log")
ax.set_xlabel("Lokalisierungsfehler [m]")
ax.set_ylabel("Anteil der Anfragen")
ax.set_title(f"{EMBEDDING_NAME}  |  {len(query_metadata):,} Anfragen")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURE_DIR / f"{EMBEDDING_NAME}_lokalisierungsfehler.png", bbox_inches="tight")
print(f"gespeichert: {FIGURE_DIR / (EMBEDDING_NAME + '_lokalisierungsfehler.png')}")
plt.show()

In [ ]:
EVAL_PATH = RESULT_DIR / "localization" / f"{EMBEDDING_NAME}.json"
EVAL_PATH.parent.mkdir(parents=True, exist_ok=True)
EVAL_PATH.write_text(
    json.dumps(
        {
            "datum": pd.Timestamp.now().strftime("%Y-%m-%d"),
            "method": METHOD,
            "adapter": ADAPTER,
            "embedding_name": EMBEDDING_NAME,
            "fingerprint_hash": short_hash(FINGERPRINT),
            "code_version": code_version(PROJECT_ROOT),
            "top_k": TOP_K,
            "n_queries": int(len(query_metadata)),
            "verfahren": tabelle,
            "fehlstruktur_top1": fehlstruktur,
        },
        indent=2,
    )
)
print(f"gespeichert: {EVAL_PATH}")